In [1]:
import random
from sqlalchemy import create_engine, Table, MetaData, Column, Integer, String, TIMESTAMP, text, BigInteger, Numeric, UniqueConstraint, Index
from sqlalchemy.sql import select
from sqlalchemy.exc import SQLAlchemyError
from datetime import datetime, timezone

# Database credentials (Target Database)
DB_HOST = "127.0.0.1"
DB_PORT = "3306"
DB_USER = "ndadmin"
DB_PASSWORD = "ndADMIN%402025"

source_engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/emd_serono")
dest_engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/emd_serono_mapping")

metadata = MetaData()

In [2]:
with source_engine.connect() as conn:
    query = text("""
    select u.uid, e.date, e.encounterID, u.cdate from users as u
    left join enc as e
    on u.uid = e.patientID
    where u.UserType = 3
    order by 1, 2, 3
    """)
    result = conn.execute(query)
    data = result.fetchall()

print("length of data ", len(data))

length of data  567193


In [3]:
patient_mapping_table = Table('patient_mapping_table', metadata,
    Column('nd_patient_id', BigInteger, primary_key=True, autoincrement=True),
    Column('patient_id', Integer, nullable=False),
    Column('offset', Integer, nullable=False),
    Column('registration_date', TIMESTAMP, nullable=True),
    Column('reference_mapping', Integer, nullable=True),
    Column('created_by', String(50), nullable=False),
    Column('created_at', TIMESTAMP, nullable=False),
    Column('updated_by', String(50), nullable=False),
    Column('updated_at', TIMESTAMP, nullable=False),
    UniqueConstraint('patient_id', name='uq_patient_id'),
    Index('ix_patient_id', 'patient_id')
)

metadata.create_all(dest_engine)
print("patient_mapping_table has been created successfully")

with dest_engine.connect() as conn:
    conn.execute(text("ALTER TABLE patient_mapping_table AUTO_INCREMENT = 100100000000001;"))

encounter_mapping_table = Table('encounter_mapping_table', metadata,
    Column('id', Integer, primary_key=True, autoincrement=True),
    Column('patient_id', Integer, nullable=False),
    Column('encounter_id', Integer, nullable=False),
    Column('nd_encounter_id', Numeric(20, 0), nullable=False),
    Column('created_by', String(50), nullable=False),
    Column('created_at', TIMESTAMP, nullable=False),
    Column('updated_by', String(50), nullable=False),
    Column('updated_at', TIMESTAMP, nullable=False),
    UniqueConstraint('nd_encounter_id', name='uq_nd_encounter_id'),
    UniqueConstraint('encounter_id', name='uq_encounter_id'),
    Index('ix_patient_id', 'patient_id')
)

metadata.create_all(dest_engine)
print("patientid_visitid_mapping has been created successfully")

patient_mapping_table has been created successfully
patientid_visitid_mapping has been created successfully


In [4]:
batch_size = 10000
total_batches = len(data) // batch_size + 1
created_by = 'nd-admin'
created_at = datetime.now(timezone.utc)
updated_by = 'nd-admin'
updated_at = datetime.now(timezone.utc)

# Batch insert logic with proper transaction handling
with dest_engine.connect() as conn:
    existing_patient_ids = {}
    patient_encounters = {}

    # Define the list of possible offset values
    possible_offsets = list(range(-38, -29)) + list(range(30, 39))

    for batch_number in range(total_batches):
        # Get the current batch of data
        batch_start = batch_number * batch_size
        batch_end = min((batch_number + 1) * batch_size, len(data))
        batch_data = data[batch_start:batch_end]

        patients_data = []
        encounters_data = []

        trans = conn.begin()

        for row in batch_data:
            patient_id = row[0]
            registration_date = row[3]

            if patient_id not in existing_patient_ids:
                offset = random.choice(possible_offsets)
                patients_data.append({
                    'patient_id': patient_id,
                    'offset': offset,
                    'registration_date': registration_date,
                    'reference_mapping': None,
                    'created_by': created_by,
                    'created_at': created_at,
                    'updated_by': updated_by,
                    'updated_at': updated_at
                })
                existing_patient_ids[patient_id] = {'offset': offset}
        
        try:
            if patients_data:
                result = conn.execute(patient_mapping_table.insert(), patients_data)

            for row in batch_data:
                patient_id = row[0]
                encounter_id = row[2]

                if encounter_id:
                    nd_patient_id_query = patient_mapping_table.select().where(
                        patient_mapping_table.c.patient_id == patient_id
                    )
                    nd_patient_id = conn.execute(nd_patient_id_query).scalar()

                    if not nd_patient_id:
                        raise ValueError(f"nd_patient_id not found for patient_id: {patient_id}")
                    
                    if patient_id not in patient_encounters:
                        encid_counter = 1
                    else:
                        encid_counter += 1

                    # Generate Encounter ID
                    nd_encounter_id = f"{nd_patient_id}{encid_counter:04d}"

                    patient_encounters[patient_id] = [nd_encounter_id, nd_encounter_id]

                    encounters_data.append({
                        'patient_id': patient_id,
                        'encounter_id': encounter_id,
                        'nd_encounter_id': nd_encounter_id,
                        'created_by': created_by,
                        'created_at': created_at,
                        'updated_by': updated_by,
                        'updated_at': updated_at
                    })

            # Insert the visit data into `patientid_visitid_mapping_table`
            if encounters_data:
                conn.execute(encounter_mapping_table.insert(), encounters_data)

            trans.commit()
            print(f"Batch {batch_number + 1}/{total_batches} inserted successfully.")
        except SQLAlchemyError as e:
            trans.rollback()
            print(f"Error in batch {batch_number + 1}/{total_batches}: {e}")
            break

print("All batches has been processed successfully.")

Batch 1/57 inserted successfully.
Batch 2/57 inserted successfully.
Batch 3/57 inserted successfully.
Batch 4/57 inserted successfully.
Batch 5/57 inserted successfully.
Batch 6/57 inserted successfully.
Batch 7/57 inserted successfully.
Batch 8/57 inserted successfully.
Batch 9/57 inserted successfully.
Batch 10/57 inserted successfully.
Batch 11/57 inserted successfully.
Batch 12/57 inserted successfully.
Batch 13/57 inserted successfully.
Batch 14/57 inserted successfully.
Batch 15/57 inserted successfully.
Batch 16/57 inserted successfully.
Batch 17/57 inserted successfully.
Batch 18/57 inserted successfully.
Batch 19/57 inserted successfully.
Batch 20/57 inserted successfully.
Batch 21/57 inserted successfully.
Batch 22/57 inserted successfully.
Batch 23/57 inserted successfully.
Batch 24/57 inserted successfully.
Batch 25/57 inserted successfully.
Batch 26/57 inserted successfully.
Batch 27/57 inserted successfully.
Batch 28/57 inserted successfully.
Batch 29/57 inserted successf